In [0]:
# Transform the data, spilt the table into three, 2 dimension, one fact for the star schema and also make it ACID to friendly

In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql.functions import lit, col, expr, current_timestamp, to_timestamp, sha2, concat_ws, coalesce, monotonically_increasing_id
# from delta.tables import DeltaTable
# from pyspark.sql import Window

In [0]:
# PySpark built-in functions (F.col, F.sum, F.avg etc.)
# Used to transform and manipulate dataframe columns
from pyspark.sql import functions as F

# Specific functions imported individually for cleaner code:
from pyspark.sql.functions import (
    lit,                      # Adds a constant/fixed value as a column
    col,                      # References a column by name
    expr,                     # Runs SQL expressions inside PySpark
    current_timestamp,        # Gets the current date and time
    to_timestamp,             # Converts string to timestamp format
    sha2,                     # Encrypts sensitive data (e.g. patient ID) using SHA-256 hashing
    concat_ws,                # Joins multiple columns into one string with a separator
    coalesce,                 # Returns first non-null value from a list of columns
    monotonically_increasing_id # Generates a unique ID number for each row
)

# Gives access to Delta Lake features
# Used for merge, update, delete operations on Delta tables
from delta.tables import DeltaTable

# Used to define window operations
# e.g. ranking, running totals, lag/lead across rows
from pyspark.sql import Window

In [ ]:
#ADLS configuration (access to the storage account)
spark.conf.set(
  "fs.azure.account.key.Storageaccount_name.dfs.core.windows.net",
  <<"Storage_Access_Key">>
)



#specifying the part to write the data
bronze_path = "abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path"

##patient_flow is a new container I just created that will be a folder inside bronze container

In [0]:
#ADLS configuration (access to the storage account)
spark.conf.set(
  "fs.azure.account.key.hospitalstoragedon.dfs.core.windows.net",
  dbutils.secrets.get(scope = "hospitalanalyticsvaultscope", key = "storage-connection")
)

In [0]:
# Container Paths
silver_path = "abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path" # The path we read the data from
gold_dim_patient ="abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path" ## Is the part that we write dimension table for the patient into
gold_dim_department ="abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path" ## Is the part that write the dimension department table
gold_fact = "abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path" ## Is the path that write the fact table into

In [0]:
# Reading silver data (assume append-only) To read the whole dataset, to compare the old and the new record
silver_df = spark.read.format("delta").load(silver_path)

In [0]:
# Define window for latest admission per patient
w = Window.partitionBy("patient_id").orderBy(F.col("admission_time").desc())

In [0]:
#To choose the latest admission time and remove old admission time todata and assign that to the "silver_df" variable
silver_df = (
    silver_df
    .withColumn("row_num", F.row_number().over(w))  # Rank by latest admission_time
    .filter(F.col("row_num") == 1)                  # Keep only latest row
    .drop("row_num")
)

###### Patient Dimension Table Creation starts

In [0]:
# Patient Dimension Table Creation starts
# Prepare incoming dimension records (deduplicated per patient, latest record) Demographic data of patient and effective to track the status of that particular record
incoming_patient = (silver_df
                    .select("patient_id", "gender", "age")
                    .withColumn("effective_from", current_timestamp())
                   )

In [0]:
# Create target if not exists
if not DeltaTable.isDeltaTable(spark, gold_dim_patient):
    # initialize table with schema and empty data
    incoming_patient.withColumn("surrogate_key", F.monotonically_increasing_id()) \
                    .withColumn("effective_to", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("overwrite").save(gold_dim_patient)

In [0]:
# Loading target as DeltaTable
target_patient = DeltaTable.forPath(spark, gold_dim_patient)

# Creating an expression to detect attribute changes (hash or explicit comparisons)
# Using a simple concat hash to detect changes

incoming_patient = incoming_patient.withColumn(
    "_hash",
    F.sha2(F.concat_ws("||", F.coalesce(col("gender"), lit("NA")), F.coalesce(col("age").cast("string"), lit("NA"))), 256)
)

####To compare the old and the new record demographich data with old demography


# Bring target current hash
target_patient_df = spark.read.format("delta").load(gold_dim_patient).withColumn(
    "_target_hash",
    F.sha2(F.concat_ws("||", F.coalesce(col("gender"), lit("NA")), F.coalesce(col("age").cast("string"), lit("NA"))), 256)
).select("surrogate_key", "patient_id", "gender", "age", "is_current", "_target_hash", "effective_from", "effective_to")


In [0]:
# Creating a temp views for merge
incoming_patient.createOrReplaceTempView("incoming_patient_tmp")
target_patient_df.createOrReplaceTempView("target_patient_tmp")

# We'll implement in two steps using Delta MERGE (safe & explicit)


In [0]:
# 1) Mark old current rows as not current where changed
changes_df = spark.sql("""
SELECT t.surrogate_key, t.patient_id
FROM target_patient_tmp t
JOIN incoming_patient_tmp i
  ON t.patient_id = i.patient_id
WHERE t.is_current = true AND t._target_hash <> i._hash
""")

In [0]:
changed_keys = [row['surrogate_key'] for row in changes_df.collect()] ### to pull out the surrogate keys after particular records with new change to gender or age and assign the surrogate_key to changed keys

if changed_keys:
    # Update existing current records: set is_current=false and effective_to=current_timestamp()
    target_patient.update(
        condition = expr("is_current = true AND surrogate_key IN ({})".format(",".join([str(k) for k in changed_keys]))),
        set = {
            "is_current": expr("false"),
            "effective_to": expr("current_timestamp()")
        }
    )

In [0]:
# 2) Insert new rows for changed & new records
# Build insert DF: join incoming with target to figure new inserts where either not exists or changed
inserts_df = spark.sql("""
SELECT i.patient_id, i.gender, i.age, i.effective_from, i._hash
FROM incoming_patient_tmp i
LEFT JOIN target_patient_tmp t
  ON i.patient_id = t.patient_id AND t.is_current = true
WHERE t.patient_id IS NULL OR t._target_hash <> i._hash
""").withColumn("surrogate_key", F.monotonically_increasing_id()) \
  .withColumn("effective_to", lit(None).cast("timestamp")) \
  .withColumn("is_current", lit(True)) \
  .select("surrogate_key", "patient_id", "gender", "age", "effective_from", "effective_to", "is_current")

In [0]:
# Append new rows
if inserts_df.count() > 0:
    inserts_df.write.format("delta").mode("append").save(gold_dim_patient)

###### Department Dimension Table Creation

In [0]:
# Department Dimension Table Creation

# prepare incoming (latest per patient feed snapshot)
incoming_dept = (silver_df
                 .select("department", "hospital_id")
                )

In [0]:
# add hash and dedupe incoming (one row per natural key)
incoming_dept = incoming_dept.dropDuplicates(["department", "hospital_id"]) \
    .withColumn("surrogate_key", monotonically_increasing_id())


In [0]:
# initializing table if missing

incoming_dept.select("surrogate_key", "department", "hospital_id") \
    .write.format("delta").mode("overwrite").save(gold_dim_department)




In [0]:
# Creating Fact table

# Read current dims (filter is_current=true)
dim_patient_df = (spark.read.format("delta").load(gold_dim_patient)
                  .filter(col("is_current") == True)
                  .select(col("surrogate_key").alias("surrogate_key_patient"), "patient_id", "gender", "age"))

dim_dept_df = (spark.read.format("delta").load(gold_dim_department)
               .select(col("surrogate_key").alias("surrogate_key_dept"), "department", "hospital_id"))



In [0]:
# Building base fact from silver events
fact_base = (silver_df
             .select("patient_id", "department", "hospital_id", "admission_time", "discharge_time", "bed_id")
             .withColumn("admission_date", F.to_date("admission_time"))
            )

In [0]:
# performing join to get surrogate keys
fact_enriched = (fact_base
                 .join(dim_patient_df, on="patient_id", how="left")
                 .join(dim_dept_df, on=["department", "hospital_id"], how="left")
                )

In [0]:
# Compute metrics
fact_enriched = fact_enriched.withColumn("length_of_stay_hours",
                                         (F.unix_timestamp(col("discharge_time")) - F.unix_timestamp(col("admission_time"))) / 3600.0) \
                             .withColumn("is_currently_admitted", F.when(col("discharge_time") > current_timestamp(), lit(True)).otherwise(lit(False))) \
                             .withColumn("event_ingestion_time", current_timestamp())

In [0]:
# Let's make column names explicit instead:
fact_final = fact_enriched.select(
    F.monotonically_increasing_id().alias("fact_id"),
    col("surrogate_key_patient").alias("patient_sk"),
    col("surrogate_key_dept").alias("department_sk"),
    "admission_time",
    "discharge_time",
    "admission_date",
    "length_of_stay_hours",
    "is_currently_admitted",
    "bed_id",
    "event_ingestion_time"
)

In [0]:
# Persist fact table partitioned by admission_date (helps Synapse / queries)
fact_final.write.format("delta").mode("overwrite").save(gold_fact)

In [0]:
# Quick sanity checks
print("Patient dim count:", spark.read.format("delta").load(gold_dim_patient).count())
print("Department dim count:", spark.read.format("delta").load(gold_dim_department).count())
print("Fact rows:", spark.read.format("delta").load(gold_fact).count())

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5792560231732470>, line 2
      1 # Quick sanity checks
----> 2 print("Patient dim count:", spark.read.format("delta").load(gold_dim_patient).count())
      3 print("Department dim count:", spark.read.format("delta").load(gold_dim_department).count())
      4 print("Fact rows:", spark.read.format("delta").load(gold_fact).count())

NameError: name 'gold_dim_patient' is not defined

In [0]:
display(spark.read.format("delta").load(gold_dim_patient))

patient_id,gender,age,effective_from,surrogate_key,effective_to,is_current
0009a17c-e436-4e50-bcd5-aff7c3cc86c5,Male,61,2026-04-19T16:09:25.241084Z,0,null,true
003f43be-fd5f-440e-b710-ce35f778bac7,Male,85,2026-04-19T16:09:25.241084Z,1,null,true
005c6b7b-c463-4dbe-94c6-ada518896c7f,Female,74,2026-04-19T16:09:25.241084Z,2,null,true
00611bdc-9de2-4ced-804c-97cf87604555,Female,58,2026-04-19T16:09:25.241084Z,3,null,true
0073b0f9-8c70-4d12-bb97-0852421f55b5,Female,49,2026-04-19T16:09:25.241084Z,4,null,true
007c6054-13c5-4667-9ad8-4abcd786ceb5,Male,63,2026-04-19T16:09:25.241084Z,5,null,true
009f2d13-fa1d-4cc5-86dd-e9fc4fe324d0,Male,62,2026-04-19T16:09:25.241084Z,6,null,true
00a0bc40-80cf-4989-9cf9-a1f32fc9f2aa,Female,94,2026-04-19T16:09:25.241084Z,7,null,true
00a3b676-7e04-4975-a557-ed8ce3ef192e,Male,49,2026-04-19T16:09:25.241084Z,8,null,true
00b8e7f0-3980-4635-ba87-ec0294649c11,Male,3,2026-04-19T16:09:25.241084Z,9,null,true


In [0]:
display(spark.read.format("delta").load(gold_dim_department))

surrogate_key,department,hospital_id
0,Pediatrics,5
1,Surgery,3
2,Surgery,2
3,ICU,3
4,Emergency,5
5,Cardiology,1
6,Emergency,7
7,Surgery,1
8,Oncology,3
9,Emergency,6


In [0]:
display(spark.read.format("delta").load(gold_fact))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5792560231732472>, line 1
----> 1 display(spark.read.format("delta").load(gold_fact))

NameError: name 'gold_fact' is not defined

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5792560231732474>, line 1
----> 1 display(spark.read.format("delta").load(gold_fact))

NameError: name 'gold_fact' is not defined

In [0]:
#ADLS configuration (access to the storage account)
spark.conf.set(
  "fs.azure.account.key.hospitalstoragedon.dfs.core.windows.net",
  dbutils.secrets.get(scope = "hospitalanalyticsvaultscope", key = "storage-connection")
)
gold_dim_patient = "abfss://gold@hospitalstoragedon.dfs.core.windows.net/dim_patient" ## Is the part that we write dimension table for the patient into

display(spark.read.format("delta").load(gold_dim_patient))

patient_id,gender,age,effective_from,surrogate_key,effective_to,is_current
0009a17c-e436-4e50-bcd5-aff7c3cc86c5,Male,61,2026-04-19T16:09:25.241084Z,0,null,true
003f43be-fd5f-440e-b710-ce35f778bac7,Male,85,2026-04-19T16:09:25.241084Z,1,null,true
005c6b7b-c463-4dbe-94c6-ada518896c7f,Female,74,2026-04-19T16:09:25.241084Z,2,null,true
00611bdc-9de2-4ced-804c-97cf87604555,Female,58,2026-04-19T16:09:25.241084Z,3,null,true
0073b0f9-8c70-4d12-bb97-0852421f55b5,Female,49,2026-04-19T16:09:25.241084Z,4,null,true
007c6054-13c5-4667-9ad8-4abcd786ceb5,Male,63,2026-04-19T16:09:25.241084Z,5,null,true
009f2d13-fa1d-4cc5-86dd-e9fc4fe324d0,Male,62,2026-04-19T16:09:25.241084Z,6,null,true
00a0bc40-80cf-4989-9cf9-a1f32fc9f2aa,Female,94,2026-04-19T16:09:25.241084Z,7,null,true
00a3b676-7e04-4975-a557-ed8ce3ef192e,Male,49,2026-04-19T16:09:25.241084Z,8,null,true
00b8e7f0-3980-4635-ba87-ec0294649c11,Male,3,2026-04-19T16:09:25.241084Z,9,null,true


In [0]:
# Check what is actually saved
df = spark.read.format("delta").load(gold_dim_patient)
df.show(5)
print("Row count:", df.count())

+--------------------+------+---+--------------------+-------------+------------+----------+
|          patient_id|gender|age|      effective_from|surrogate_key|effective_to|is_current|
+--------------------+------+---+--------------------+-------------+------------+----------+
|0009a17c-e436-4e5...|  Male| 61|2026-04-19 16:09:...|            0|        NULL|      true|
|003f43be-fd5f-440...|  Male| 85|2026-04-19 16:09:...|            1|        NULL|      true|
|005c6b7b-c463-4db...|Female| 74|2026-04-19 16:09:...|            2|        NULL|      true|
|00611bdc-9de2-4ce...|Female| 58|2026-04-19 16:09:...|            3|        NULL|      true|
|0073b0f9-8c70-4d1...|Female| 49|2026-04-19 16:09:...|            4|        NULL|      true|
+--------------------+------+---+--------------------+-------------+------------+----------+
only showing top 5 rows
Row count: 12277


In [0]:
# Run this in your Databricks notebook
# This shows the REAL data types saved in your Delta table
spark.read.format("delta").load(gold_dim_patient).printSchema()


root
 |-- patient_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- surrogate_key: long (nullable = true)
 |-- effective_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)

